<a href="https://colab.research.google.com/github/Maniac-extraordinaire/HotPath/blob/main/HotPath.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install stable-baselines3[extra]
!pip install transformers
!pip install torch
!pip install gymnasium
!pip install datasets
!pip install wandb


'import warnings' failed; traceback:
Traceback (most recent call last):
  File "/usr/lib/python3.12/warnings.py", line 587, in <module>
    _processoptions(sys.warnoptions)
  File "/usr/lib/python3.12/warnings.py", line 211, in _processoptions
    _setoption(arg)
  File "/usr/lib/python3.12/warnings.py", line 227, in _setoption
    import re
  File "/usr/lib/python3.12/re/__init__.py", line 125, in <module>
    from . import _compiler, _parser
  File "/usr/lib/python3.12/re/_compiler.py", line 14, in <module>
    from . import _parser
  File "/usr/lib/python3.12/re/_parser.py", line 15, in <module>
    from ._constants import *
  File "/usr/lib/python3.12/re/_constants.py", line 127, in <module>
    ATCODES = _makecodes(
              ^^^^^^^^^^^
  File "/usr/lib/python3.12/re/_constants.py", line 71, in _makecodes
    globals().update({item.name: item for item in items})
KeyboardInterrupt


In [3]:
import torch
import gymnasium as gym
import transformers
import stable_baselines3

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("All imports successful ✅")

PyTorch: 2.10.0+cu128
GPU available: True
All imports successful ✅


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [12]:
import time
import numpy as np

def run_code_and_time(code: str, test_input: str = "", timeout: int = 5) -> float:
    """
    Runs code in the SAME process using exec() — 100x faster than subprocess.
    """
    try:
        namespace = {}
        start = time.perf_counter()
        exec(textwrap.dedent(code), namespace)
        end = time.perf_counter()
        return end - start
    except Exception:
        return 999.0

# Quick test
test_code = """
nums = list(range(10000))
result = [x for x in nums if x in nums]
"""
t = run_code_and_time(test_code)
print(f"Code ran in {t:.4f} seconds ✅")

Code ran in 0.5226 seconds ✅


In [13]:
import re

def apply_transformation(code: str, action: int) -> str:
    """
    Takes code and an action ID, returns transformed code.
    """
    if action == 0:  # no_op
        return code

    elif action == 1:  # list_to_set
        # Find patterns like: x in some_list → convert list to set
        transformed = re.sub(
            r'(\w+)\s*=\s*list\(([^)]+)\)',
            lambda m: f"{m.group(1)} = set({m.group(2)})",
            code
        )
        return transformed

    elif action == 2:  # add_memoization
        # Wrap any recursive function with a cache
        if "def " in code:
            transformed = "from functools import lru_cache\n" + code
            transformed = re.sub(r'(def \w+\()', r'@lru_cache(maxsize=None)\n\1', transformed)
            return transformed
        return code

    elif action == 3:  # use_enumerate
        # Replace for i in range(len(x)) with enumerate
        transformed = re.sub(
            r'for (\w+) in range\(len\((\w+)\)\)',
            r'for \1, _ in enumerate(\2)',
            code
        )
        return transformed

    elif action == 4:  # use_zip
        return code  # placeholder for now

    return code

# Test it
slow_code = """
nums = list(range(10000))
result = [x for x in nums if x in nums]
"""

fast_code = apply_transformation(slow_code, action=1)
print("=== Transformed Code ===")
print(fast_code)

t_slow = run_code_and_time(slow_code)
t_fast = run_code_and_time(fast_code)
print(f"\nSlow: {t_slow:.4f}s")
print(f"Fast: {t_fast:.4f}s")
print(f"Speedup: {t_slow/t_fast:.2f}x 🚀")

=== Transformed Code ===

nums = set(range(10000))
result = [x for x in nums if x in nums]


Slow: 0.5437s
Fast: 0.0011s
Speedup: 517.14x 🚀


In [15]:
import gymnasium as gym
import numpy as np
from gymnasium import spaces

class CodeOptimizerEnv(gym.Env):
    """
    RL Environment where:
    - State  = a code snippet (represented as a fixed vector)
    - Action = which transformation to apply (0-4)
    - Reward = speedup ratio after transformation
    """

    def __init__(self, code_problems: list):
        super().__init__()

        self.code_problems = code_problems      # list of slow code snippets
        self.current_code = None
        self.original_code = None
        self.step_count = 0
        self.max_steps = 5                      # agent gets 5 attempts per problem

        # Action space: 5 possible transformations
        self.action_space = spaces.Discrete(len(TRANSFORMATIONS))

        # Observation space: we encode code as a 768-dim vector (CodeBERT size)
        # For now we use a simple placeholder vector
        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(768,),
            dtype=np.float32
        )

    def _encode_code(self, code: str) -> np.ndarray:
        """
        Converts code string to a fixed-size vector.
        Placeholder for now — will plug in CodeBERT later.
        """
        # Simple hash-based encoding for now
        vec = np.zeros(768, dtype=np.float32)
        for i, ch in enumerate(code[:768]):
            vec[i % 768] += ord(ch) / 1000.0
        return vec / (np.linalg.norm(vec) + 1e-8)  # normalize

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # Pick a random problem each episode
        self.original_code = np.random.choice(self.code_problems)
        self.current_code = self.original_code
        self.step_count = 0
        obs = self._encode_code(self.current_code)
        return obs, {}

    def step(self, action: int):
        self.step_count += 1

        # Apply transformation
        new_code = apply_transformation(self.current_code, action)

        # Benchmark old vs new
        t_old = run_code_and_time(self.current_code)
        t_new = run_code_and_time(new_code)

        # Calculate reward
        if t_new < t_old and t_new < 999:
            reward = t_old / t_new   # speedup ratio (e.g. 9.7)
        elif t_new == t_old:
            reward = 0.0             # no change
        else:
            reward = -1.0            # got slower or crashed

        # Update current code to the better version
        if reward > 0:
            self.current_code = new_code

        # Done after max_steps
        done = self.step_count >= self.max_steps

        obs = self._encode_code(self.current_code)
        info = {
            "action": TRANSFORMATIONS[action],
            "speedup": t_old / (t_new + 1e-8),
            "current_code": self.current_code
        }

        return obs, reward, done, False, info

    def render(self):
        print(f"Step {self.step_count} | Current code:\n{self.current_code}")

In [16]:
# Sample slow code problems for training
PROBLEMS = [
    """
nums = list(range(10000))
result = [x for x in nums if x in nums]
""",
    """
nums = list(range(5000))
target = 4999
found = target in nums
""",
    """
data = list(range(8000))
unique = [x for x in data if data.count(x) == 1]
""",
]

# Create environment
env = CodeOptimizerEnv(PROBLEMS)

# Manual test run
obs, info = env.reset()
print(f"Observation shape: {obs.shape}")
print(f"Action space: {env.action_space}")
print()

# Simulate agent taking actions
for step in range(5):
    action = env.action_space.sample()  # random action for now
    obs, reward, done, truncated, info = env.step(action)
    print(f"Step {step+1} | Action: {info['action']:<20} | Reward: {reward:.3f} | Speedup: {info['speedup']:.2f}x")
    if done:
        break

print("\nEnvironment working ✅")

Observation shape: (768,)
Action space: Discrete(5)

Step 1 | Action: list_to_set          | Reward: -1.000 | Speedup: 0.68x
Step 2 | Action: list_to_set          | Reward: -1.000 | Speedup: 0.90x
Step 3 | Action: add_memoization      | Reward: -1.000 | Speedup: 0.99x
Step 4 | Action: add_memoization      | Reward: 1.103 | Speedup: 1.10x
Step 5 | Action: use_enumerate        | Reward: -1.000 | Speedup: 0.94x

Environment working ✅


In [21]:
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

env = Monitor(CodeOptimizerEnv(PROBLEMS))

model = PPO(
    policy="MlpPolicy",
    env=env,
    verbose=1,
    learning_rate=3e-4,
    n_steps=32,
    batch_size=8,
    n_epochs=3,
    gamma=0.95,
)

print("Starting training...")
model.learn(total_timesteps=2000)  # sweet spot
print("\nTraining complete ✅")

Using cuda device
Wrapping the env in a DummyVecEnv.
Starting training...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 5        |
|    ep_rew_mean     | 194      |
| time/              |          |
|    fps             | 1        |
|    iterations      | 1        |
|    time_elapsed    | 17       |
|    total_timesteps | 32       |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 5            |
|    ep_rew_mean          | 294          |
| time/                   |              |
|    fps                  | 1            |
|    iterations           | 2            |
|    time_elapsed         | 39           |
|    total_timesteps      | 64           |
| train/                  |              |
|    approx_kl            | 5.517155e-06 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         |

KeyboardInterrupt: 

In [19]:
from stable_baselines3 import PPO

# Load model (or use existing `model` from above)
# model = PPO.load("/content/drive/MyDrive/code_optimizer_ppo")

print("=" * 55)
print("       EVALUATING TRAINED AGENT")
print("=" * 55)

total_reward = 0
n_episodes = 5

for ep in range(n_episodes):
    obs, _ = env.reset()
    ep_reward = 0
    print(f"\n📘 Episode {ep+1}")
    print(f"Original code:{env.unwrapped.original_code}")

    for step in range(5):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, _, info = env.step(int(action))
        ep_reward += reward
        print(f"  Step {step+1} | Action: {info['action']:<20} | Reward: {reward:.3f} | Speedup: {info['speedup']:.2f}x")
        if done:
            break

    print(f"  Episode reward: {ep_reward:.3f}")
    total_reward += ep_reward

print(f"\n{'='*55}")
print(f"Average reward over {n_episodes} episodes: {total_reward/n_episodes:.3f}")

       EVALUATING TRAINED AGENT

📘 Episode 1
Original code:
nums = list(range(5000))
target = 4999
found = target in nums

  Step 1 | Action: use_zip              | Reward: 1.314 | Speedup: 1.31x
  Step 2 | Action: use_zip              | Reward: 1.357 | Speedup: 1.36x
  Step 3 | Action: use_zip              | Reward: 1.343 | Speedup: 1.34x
  Step 4 | Action: use_zip              | Reward: 1.277 | Speedup: 1.28x
  Step 5 | Action: use_zip              | Reward: 1.317 | Speedup: 1.32x
  Episode reward: 6.608

📘 Episode 2
Original code:
nums = list(range(10000))
result = [x for x in nums if x in nums]

  Step 1 | Action: use_zip              | Reward: -1.000 | Speedup: 0.97x
  Step 2 | Action: use_zip              | Reward: -1.000 | Speedup: 0.95x
  Step 3 | Action: use_zip              | Reward: -1.000 | Speedup: 0.95x
  Step 4 | Action: use_zip              | Reward: -1.000 | Speedup: 0.95x
  Step 5 | Action: use_zip              | Reward: -1.000 | Speedup: 0.99x
  Episode reward: -5.00

In [22]:
from google.colab import drive
drive.mount('/content/drive')

# Save model
model.save("/content/drive/MyDrive/code_optimizer_ppo")
print("Model saved ✅")

Mounted at /content/drive
Model saved ✅
